In [1]:
import pandas as pd

In [2]:
df1 = pd.read_csv('C:\\Users\\004NQ8744\\OneDrive\\Roadwork\\manpowergroup\\text2sqlbot\\data\\employee_data.csv')
df2 = pd.read_csv('C:\\Users\\004NQ8744\\OneDrive\\Roadwork\\manpowergroup\\text2sqlbot\\data\\employee_engagement_survey_data.csv')
df3 = pd.read_csv('C:\\Users\\004NQ8744\\OneDrive\\Roadwork\\manpowergroup\\text2sqlbot\\data\\training_and_development_data.csv')

In [3]:
df1.head()

,EmpID,FirstName,LastName,StartDate,ExitDate,Title,Supervisor,ADEmail,BusinessUnit,EmployeeStatus,...,Division,DOB,State,JobFunctionDescription,GenderCode,LocationCode,RaceDesc,MaritalDesc,Performance Score,Current Employee Rating
0,3427,Uriah,Bridges,20-Sep-19,NaN,Production Technician I,Peter Oneill,uriah.bridges@bilearner.com,CCDR,Active,...,Finance & Accounting,07-10-1969,MA,Accounting,Female,34904,White,Widowed,Fully Meets,4
1,3428,Paula,Small,11-Feb-23,NaN,Production Technician I,Renee Mccormick,paula.small@bilearner.com,EW,Active,...,Aerial,30-08-1965,MA,Labor,Male,6593,Hispanic,Widowed,Fully Meets,3
2,3429,Edward,Buck,10-Dec-18,NaN,Area Sales Manager,Crystal Walker,edward.buck@bilearner.com,PL,Active,...,General - Sga,06-10-1991,MA,Assistant,Male,2330,Hispanic,Widowed,Fully Meets,4
3,3430,Michael,Riordan,21-Jun-21,NaN,Area Sales Manager,Rebekah Wright,michael.riordan@bilearner.com,CCDR,Active,...,Finance & Accounting,04-04-1998,ND,Clerk,Male,58782,Other,Single,Fully Meets,2
4,3431,Jasmine,Onque,29-Jun-19,NaN,Area Sales Manager,Jason Kim,jasmine.onque@bilearner.com,TNS,Active,...,General - Con,29-08-1969,FL,Laborer,Female,33174,Other,Married,Fully Meets,3


In [5]:
df1.columns

Index(['EmpID', 'FirstName', 'LastName', 'StartDate', 'ExitDate', 'Title',
       'Supervisor', 'ADEmail', 'BusinessUnit', 'EmployeeStatus',
       'EmployeeType', 'PayZone', 'EmployeeClassificationType',
       'TerminationType', 'TerminationDescription', 'DepartmentType',
       'Division', 'DOB', 'State', 'JobFunctionDescription', 'GenderCode',
       'LocationCode', 'RaceDesc', 'MaritalDesc', 'Performance Score',
       'Current Employee Rating'],
      dtype='str')

In [7]:
df1["DOB"].dtype

<StringDtype(na_value=nan)>

In [33]:
df2.head()

,Employee ID,Survey Date,Engagement Score,Satisfaction Score,Work-Life Balance Score
0,1001,10-10-2022,2,5,5
1,1002,03-08-2023,4,5,3
2,1003,03-01-2023,2,5,2
3,1004,30-07-2023,3,5,3
4,1005,19-06-2023,2,4,5


In [34]:
df1.columns

Index(['EmpID', 'FirstName', 'LastName', 'StartDate', 'ExitDate', 'Title',
       'Supervisor', 'ADEmail', 'BusinessUnit', 'EmployeeStatus',
       'EmployeeType', 'PayZone', 'EmployeeClassificationType',
       'TerminationType', 'TerminationDescription', 'DepartmentType',
       'Division', 'DOB', 'State', 'JobFunctionDescription', 'GenderCode',
       'LocationCode', 'RaceDesc', 'MaritalDesc', 'Performance Score',
       'Current Employee Rating'],
      dtype='str')

In [35]:
df2.columns

Index(['Employee ID', 'Survey Date', 'Engagement Score', 'Satisfaction Score',
       'Work-Life Balance Score'],
      dtype='str')

In [36]:
df3.columns

Index(['Employee ID', 'Training Date', 'Training Program Name',
       'Training Type', 'Training Outcome', 'Location', 'Trainer',
       'Training Duration(Days)', 'Training Cost'],
      dtype='str')

In [13]:
import re

In [21]:
"Employee_Data#1.csv".rsplit(".", 1)[0].lower().strip()

'employee_data#1'

In [22]:
re.sub(r"[^a-z0-9_]", "_", 'employee_data1')

'employee_data1'

In [30]:
re.sub(r"_+", "_", "0__employee___________data1").strip("_")

'0_employee_data1'

In [29]:
name = '0_employee_data1'
if name and name[0].isdigit():
    name = "t_" + name
name

't_0_employee_data1'

In [8]:
DATE_FORMATS = [
        "%d-%m-%Y",
        "%d/%m/%Y",
        "%Y-%m-%d",
        "%m/%d/%Y",
        "%m-%d-%Y",
        "%d-%b-%Y",
        "%B %d, %Y",
    ]

In [9]:
for col in df1.columns:
    for fmt in DATE_FORMATS:
        try:
            pd.to_datetime(df1[col], format=fmt)
            print(f"Column '{col}' matches date format: {fmt}")
            break
        except (ValueError, TypeError):
            continue

Column 'DOB' matches date format: %d-%m-%Y


In [14]:
def token_overlap(c1: str, c2: str) -> float:
    """Jaccard-style token overlap between two column names."""
    def tokenize(s):
        s = re.sub(r"[^a-z0-9]", " ", s.lower())
        tokens = set(s.split())
        # Also add substrings of length 3+ to catch 'emp' in 'employee'
        for t in list(tokens):
            if len(t) >= 4:
                for size in range(3, len(t)):
                    tokens.add(t[:size])
        return tokens

    t1, t2 = tokenize(c1), tokenize(c2)
    if not t1 or not t2:
        return 0.0
    intersection = t1 & t2
    union = t1 | t2
    return len(intersection) / len(union)


In [15]:
token_overlap("emp", "employee_id")

0.14285714285714285

In [17]:
import pandas as pd

In [18]:
def sanitize_table_name(filename: str) -> str:
    name = filename.rsplit(".", 1)[0]
    name = name.lower().strip()
    name = re.sub(r"[^a-z0-9_]", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    if name and name[0].isdigit():
        name = "t_" + name
    return name or "table"

In [25]:
columns = [
    {"name": col, "type": str(dtype)}
    for col, dtype in df1.dtypes.items()
]

In [26]:
columns

[{'name': 'EmpID', 'type': 'int64'},
 {'name': 'FirstName', 'type': 'str'},
 {'name': 'LastName', 'type': 'str'},
 {'name': 'StartDate', 'type': 'str'},
 {'name': 'ExitDate', 'type': 'str'},
 {'name': 'Title', 'type': 'str'},
 {'name': 'Supervisor', 'type': 'str'},
 {'name': 'ADEmail', 'type': 'str'},
 {'name': 'BusinessUnit', 'type': 'str'},
 {'name': 'EmployeeStatus', 'type': 'str'},
 {'name': 'EmployeeType', 'type': 'str'},
 {'name': 'PayZone', 'type': 'str'},
 {'name': 'EmployeeClassificationType', 'type': 'str'},
 {'name': 'TerminationType', 'type': 'str'},
 {'name': 'TerminationDescription', 'type': 'str'},
 {'name': 'DepartmentType', 'type': 'str'},
 {'name': 'Division', 'type': 'str'},
 {'name': 'DOB', 'type': 'str'},
 {'name': 'State', 'type': 'str'},
 {'name': 'JobFunctionDescription', 'type': 'str'},
 {'name': 'GenderCode', 'type': 'str'},
 {'name': 'LocationCode', 'type': 'int64'},
 {'name': 'RaceDesc', 'type': 'str'},
 {'name': 'MaritalDesc', 'type': 'str'},
 {'name': 'Per

In [27]:
tables = {}
for f in [open("C:\\Users\\004NQ8744\\OneDrive\\Roadwork\\manpowergroup\\text2sqlbot\\data\\employee_data.csv"), open("C:\\Users\\004NQ8744\\OneDrive\\Roadwork\\manpowergroup\\text2sqlbot\\data\\employee_engagement_survey_data.csv"), open("C:\\Users\\004NQ8744\\OneDrive\\Roadwork\\manpowergroup\\text2sqlbot\\data\\training_and_development_data.csv")]:
    try:
        df = pd.read_csv(f)
        df.columns = [
            re.sub(r"[^a-z0-9_]", "_", c.lower().strip()) for c in df.columns
        ]
        df.columns = [re.sub(r"_+", "_", c).strip("_") for c in df.columns]

        table_name = sanitize_table_name(f.name)

        sample_df = df.head(3)
        sample_rows = sample_df.to_string(index=False)

        columns = [
            {"name": col, "type": str(dtype)}
            for col, dtype in df.dtypes.items()
        ]

        tables[table_name] = {
            "columns": columns,
            "sample_rows": sample_rows,
            "row_count": len(df),
            "original_filename": f.name,
        }
    except Exception as e:
        raise RuntimeError(f"Failed to load {f.name}: {e}")

In [28]:
tables

{'c_users_004nq8744_onedrive_roadwork_manpowergroup_text2sqlbot_data_employee_data': {'columns': [{'name': 'empid',
    'type': 'int64'},
   {'name': 'firstname', 'type': 'str'},
   {'name': 'lastname', 'type': 'str'},
   {'name': 'startdate', 'type': 'str'},
   {'name': 'exitdate', 'type': 'str'},
   {'name': 'title', 'type': 'str'},
   {'name': 'supervisor', 'type': 'str'},
   {'name': 'ademail', 'type': 'str'},
   {'name': 'businessunit', 'type': 'str'},
   {'name': 'employeestatus', 'type': 'str'},
   {'name': 'employeetype', 'type': 'str'},
   {'name': 'payzone', 'type': 'str'},
   {'name': 'employeeclassificationtype', 'type': 'str'},
   {'name': 'terminationtype', 'type': 'str'},
   {'name': 'terminationdescription', 'type': 'str'},
   {'name': 'departmenttype', 'type': 'str'},
   {'name': 'division', 'type': 'str'},
   {'name': 'dob', 'type': 'str'},
   {'name': 'state', 'type': 'str'},
   {'name': 'jobfunctiondescription', 'type': 'str'},
   {'name': 'gendercode', 'type': 'str

In [21]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

c:\Users\004NQ8744\OneDrive\Roadwork\manpowergroup\text2sqlbot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3497.06it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
import numpy as np
from typing import Any

In [23]:
def _types_compatible(t1: str, t2: str) -> bool:
    numeric = {"integer", "real", "numeric", "float", "int", "bigint", "double"}
    text = {"text", "varchar", "char", "string", "nvarchar"}
    t1, t2 = t1.lower(), t2.lower()
    if t1 == t2:
        return True
    if t1 in numeric and t2 in numeric:
        return True
    if t1 in text and t2 in text:
        return True
    return False

In [29]:
table_names = list(tables.keys())
relationships = []

col_embeddings: dict[str, dict[str, Any]] = {}
for tname in table_names:
    for col in tables[tname]["columns"]:
        cname = col["name"]
        key = f"{tname}.{cname}"
        col_embeddings[key] = {
            "table": tname,
            "col": cname,
            "type": col["type"],
            "embedding": model.encode(
                f"{tname} {cname}".replace("_", " "), normalize_embeddings=True
            ),
        }

keys = list(col_embeddings.keys())
for i, k1 in enumerate(keys):
    m1 = col_embeddings[k1]
    for k2 in keys[i + 1:]:
        m2 = col_embeddings[k2]
        if m1["table"] == m2["table"]:
            continue
        if not _types_compatible(m1["type"], m2["type"]):
            continue

        sem_score = float(np.dot(m1["embedding"], m2["embedding"]))
        tok_score = token_overlap(m1["col"], m2["col"])
        combined = 0.6 * sem_score + 0.4 * tok_score

        if combined >= 0.55:
            relationships.append({
                "table_a": m1["table"],
                "col_a": m1["col"],
                "table_b": m2["table"],
                "col_b": m2["col"],
                "score": round(combined, 3),
            })

# Deduplicate: keep highest-scoring pair per table combination
seen: dict[tuple, dict] = {}
for rel in sorted(relationships, key=lambda r: r["score"], reverse=True):
    pair = tuple(sorted([rel["table_a"], rel["table_b"]]))
    col_pair = (rel["col_a"], rel["col_b"])
    dedup_key = (pair, col_pair)
    if dedup_key not in seen:
        seen[dedup_key] = rel

print(list(seen.values()))

[{'table_a': 'c_users_004nq8744_onedrive_roadwork_manpowergroup_text2sqlbot_data_employee_engagement_survey_data', 'col_a': 'employee_id', 'table_b': 'c_users_004nq8744_onedrive_roadwork_manpowergroup_text2sqlbot_data_training_and_development_data', 'col_b': 'employee_id', 'score': 0.936}, {'table_a': 'c_users_004nq8744_onedrive_roadwork_manpowergroup_text2sqlbot_data_employee_data', 'col_a': 'current_employee_rating', 'table_b': 'c_users_004nq8744_onedrive_roadwork_manpowergroup_text2sqlbot_data_employee_engagement_survey_data', 'col_b': 'employee_id', 'score': 0.653}, {'table_a': 'c_users_004nq8744_onedrive_roadwork_manpowergroup_text2sqlbot_data_employee_data', 'col_a': 'current_employee_rating', 'table_b': 'c_users_004nq8744_onedrive_roadwork_manpowergroup_text2sqlbot_data_training_and_development_data', 'col_b': 'employee_id', 'score': 0.644}, {'table_a': 'c_users_004nq8744_onedrive_roadwork_manpowergroup_text2sqlbot_data_employee_data', 'col_a': 'empid', 'table_b': 'c_users_004nq

In [30]:
relationships = list(seen.values())

In [31]:

embeddings = {}
for table_name, meta in tables.items():
    cols = ", ".join(
        f"{c['name']} ({c['type']})" for c in meta["columns"]
    )
    doc = (
        f"Table: {table_name}. "
        f"Columns: {cols}. "
        f"Sample data: {meta['sample_rows'][:300]}"
    )
    embeddings[table_name] = {
        "embedding": model.encode(doc, normalize_embeddings=True),
        "doc": doc,
    }
embeddings


{'c_users_004nq8744_onedrive_roadwork_manpowergroup_text2sqlbot_data_employee_data': {'embedding': array([ 1.96490251e-02,  5.50282411e-02, -3.90596152e-03,  2.21886765e-02,
         -4.96222898e-02,  1.86872855e-02, -2.76432210e-03,  1.14368945e-02,
         -8.93702433e-02,  3.40246893e-02,  7.56848380e-02, -1.26092076e-01,
         -7.59015093e-03, -1.13704279e-01, -8.12622346e-03,  1.33697623e-02,
          1.59974992e-02,  2.17565871e-03, -1.35608260e-02, -8.72193575e-02,
         -7.11922301e-03,  2.71511357e-02, -3.92017700e-02, -2.86982898e-02,
          1.05423778e-02,  2.42040567e-02,  4.25649658e-02,  1.24483928e-01,
         -1.96021106e-02, -5.78838848e-02, -4.75884229e-02,  2.61521526e-03,
          7.96103776e-02,  2.13860963e-02,  2.39589904e-02, -3.55838388e-02,
         -2.85277255e-02,  1.20504946e-02,  4.02070768e-02,  9.14061069e-02,
         -6.17857575e-02, -6.07248917e-02,  7.52977207e-02,  2.99896710e-02,
          2.54422496e-03,  2.24902146e-02, -4.33254503e-

In [32]:
type(embeddings)

dict

In [33]:
def retrieve_relevant_tables(
    question: str,
    schema_embeddings: dict,
    tables: dict,
    top_k: int = 3,
    threshold: float = 0.15,
) -> dict:
    if not schema_embeddings:
        return tables

    if len(tables) <= 4:
        return tables

    q_emb = model.encode(question, normalize_embeddings=True)

    scores = {}
    for table_name, meta in schema_embeddings.items():
        score = float(np.dot(q_emb, meta["embedding"]))
        scores[table_name] = score

    sorted_tables = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    selected = [t for t, s in sorted_tables[:top_k] if s >= threshold]

    if not selected:
        return tables

    return {t: tables[t] for t in selected if t in tables}

In [34]:
relevant_tables = retrieve_relevant_tables(
    "Which employees have the highest engagement scores?",
    embeddings,
    tables)

In [36]:
relevant_tables.keys()

dict_keys(['c_users_004nq8744_onedrive_roadwork_manpowergroup_text2sqlbot_data_employee_data', 'c_users_004nq8744_onedrive_roadwork_manpowergroup_text2sqlbot_data_employee_engagement_survey_data', 'c_users_004nq8744_onedrive_roadwork_manpowergroup_text2sqlbot_data_training_and_development_data'])